# 🧪 07 · Measure Quality with an AI Judge — Advanced track

**Level:** Experts · **Time:** ~25 min

"Looks good" is not a metric. Here we **objectively score** generated test cases using an
**LLM-as-judge** — a second model that grades the output against a rubric. This is how teams
compare prompts, models, or fine-tunes with numbers instead of vibes.

> ⚙️ Enable GPU: *Runtime → Change runtime type → T4 GPU*.

### Step 1 — Setup

In [ ]:
!pip install -q -U transformers accelerate
import torch, json, re
from transformers import pipeline
chat = pipeline('text-generation', model='Qwen/Qwen2.5-1.5B-Instruct',
                torch_dtype=torch.bfloat16, device_map='auto')

def run(system, user, max_new_tokens=900):
    out = chat([{'role':'system','content':system}, {'role':'user','content':user}],
               max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]['generated_text'][-1]['content']

print('✅ ready')

### Step 2 — Produce two candidates to compare
We generate test cases two ways for the **same** requirement: a **weak** prompt and a
**strong** QA prompt. Then we let the judge decide which is better — objectively.

In [ ]:
REQUIREMENT = "As a user, I can pay with a credit card; invalid cards are rejected with a clear error."

WEAK = "You are a helpful assistant."
STRONG = ("You are a senior QA engineer. Produce thorough test cases covering positive, "
          "negative, edge and boundary scenarios, each with clear steps and expected results.")

ask = f"Write test cases for this requirement:\n{REQUIREMENT}"
candidate_A = run(WEAK, ask)
candidate_B = run(STRONG, ask)
print('Generated both candidates (A = weak prompt, B = strong prompt).')

### Step 3 — The judge (scores against a rubric, returns JSON)
The judge rates each candidate 1-5 on **coverage**, **correctness**, and **clarity**, with a
short reason. Forcing JSON makes the scores easy to compare.

In [ ]:
JUDGE_SYS = ('You are a strict QA lead evaluating test cases. Score from 1 (poor) to 5 '
             '(excellent). Return ONLY JSON: {"coverage":n,"correctness":n,"clarity":n,'
             '"overall":n,"reason":"..."}. Be critical and consistent.')

def judge(requirement, candidate):
    user = (f"Requirement:\n{requirement}\n\nCandidate test cases:\n{candidate}\n\n"
            "Score them now as JSON.")
    text = run(JUDGE_SYS, user, max_new_tokens=300)
    m = re.search(r'\{.*\}', text, re.DOTALL)
    return json.loads(m.group(0)) if m else {"overall": None, "reason": text}

score_A = judge(REQUIREMENT, candidate_A)
score_B = judge(REQUIREMENT, candidate_B)
print('A (weak):  ', score_A)
print('B (strong):', score_B)

### Step 4 — Compare side by side

In [ ]:
import pandas as pd
table = pd.DataFrame({'A: weak prompt': score_A, 'B: strong prompt': score_B})
table

### ⭐ Easy mode — judge your own two texts
Paste any two sets of test cases and see which scores higher.

In [ ]:
# 👇 Just run it and use the boxes.
import ipywidgets as widgets
from IPython.display import display, clear_output

ta = widgets.Textarea(placeholder='Paste candidate A...', layout=widgets.Layout(width='48%', height='150px'))
tb = widgets.Textarea(placeholder='Paste candidate B...', layout=widgets.Layout(width='48%', height='150px'))
req = widgets.Text(value=REQUIREMENT, layout=widgets.Layout(width='100%'))
go = widgets.Button(description='Judge both', button_style='primary')
out = widgets.Output()

def on_click(_):
    with out:
        clear_output()
        print('⏳ judging...')
        a = judge(req.value, ta.value or candidate_A)
        b = judge(req.value, tb.value or candidate_B)
        clear_output()
        display(pd.DataFrame({'A': a, 'B': b}))

go.on_click(on_click)
display(req, widgets.HBox([ta, tb]), go, out)

---
### 🧠 Expert discussion
- **LLM-as-judge is powerful but biased** — it can favour longer or more confident answers. Always **spot-check with a human**.
- For reliability: run the judge **multiple times and average**, and keep the rubric explicit.
- This is exactly how you'd prove whether the **fine-tuned model (Notebook 04)** actually beats the base model — judge both on the same requirements and compare the scores.
- A judge bigger/stronger than the model being judged gives more trustworthy scores.

---
### 🎮 Try this (build intuition)
From the [playground challenges](https://github.com/hn-alchemist/qa-ai-workshop/blob/main/data/playground_challenges.md) (challenge 12):
1. **Judge the judge:** run `judge()` on the *same* candidate 3 times. Do the scores match?
2. If they don't, what does that tell you about trusting a single AI evaluation?
3. Use this to compare your **fine-tuned model (Notebook 04)** against the base model on equal footing.

Discuss: when is an AI score good enough to act on, and when do you still need a human reviewer?